# Exercise 2 — Spark through Livy

Complete Exercise 1 first and stop its Spark session.

**This notebook and `02_spark_via_livy.py` are alternatives for the same
exercise. Choose one. The script is not an extra exercise or an advanced solution.**

We repeat Exercise 1's bank-loan analysis, but submit code over HTTP to
pre-deployed Livy. Your notebook is the HTTP client; the driver runs on the
Livy side and executors run on the configured Spark workers.

Run cells in order with Shift+Enter using the organiser-prepared Python kernel.
The Python companion supplies the shared helpers; importing it does not run
its terminal workflow.

**Pilot status:** live Livy validation remains pending. Do not install software
or manage containers. After an error, stop dependent cells and use Section 7
if a session ID was obtained.

## 1. Inspect the same local dataset

Reuse Exercise 1's CSV, field documentation and data-quality discussion.
This local preview does not contact Livy. Expect column names and up to five rows.
Open this notebook from the prepared Spark workspace.

In [ ]:
from pathlib import Path
import importlib.util

if globals().get('session_id') is not None:
    raise RuntimeError('Run Section 7 cleanup before restarting this exercise.')

workspace = Path.cwd()
companion = workspace / '02_spark_via_livy.py'
if not companion.is_file():
    raise FileNotFoundError('The Python companion must be beside this notebook.')

spec = importlib.util.spec_from_file_location('spark_livy_exercise', companion)
livy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(livy)  # Imports helpers only; does not call main().
session_id = None
livy.preview_local_dataset(workspace / 'data' / 'Bank_Personal_Loan_Modelling.csv')


## 2. Create a session through prepared Livy

The organiser supplies `LIVY_URL` and `LIVY_DATA_PATH`. The data path must
identify the same CSV accessible to the Livy-side driver and executors;
it need not match your local path or Exercise 1's `SPARK_DATA_PATH`.

Livy's master and driver networking are already configured. Do not guess
settings. This simple client assumes organiser-approved access without
additional API authentication; JupyterHub login does not itself secure Livy.

Session creation returns an ID, not immediate readiness. Use the returned ID,
not a hard-coded zero. Do not repeat creation while a session is recorded.

In [ ]:
if session_id is not None:
    raise RuntimeError('A session is recorded. Continue or run Section 7 cleanup.')

settings = livy.read_connection_settings()
base_url = settings['LIVY_URL']
response = livy.livy_request(base_url, 'POST', '/sessions', {'kind': 'pyspark'})
session_id = livy.response_id(response)
print('Created Livy session:', session_id)
print('Initial state:', response.get('state'))


### Wait for readiness and retrieve a first statement

The helper polls the session until idle, with a bounded wait. It then submits
a statement, uses its returned ID and polls until output is available.
Remote errors are reported rather than treated as success.

Expect the remote Spark version and master URL. Check the assigned master
with the organiser. This does not yet establish CSV access or worker execution.

In [ ]:
livy.wait_for_session(base_url, session_id)
code = "print('Spark version:', spark.version)\nprint('Spark master:', spark.sparkContext.master)"
print(livy.run_statement(base_url, session_id, code))
steps = livy.analysis_steps(settings['LIVY_DATA_PATH'])


## 3. Read CSV and inspect schema

Repeat Exercise 1's header/schema options, five-row preview and full count. Compare with Exercise 1 for the same snapshot. The variable `df` lives in the remote Livy session, not in this notebook kernel.

The cell displays the code being sent, followed by its remote output.

In [ ]:
heading, code = steps[0]
print('Code submitted to Livy:\n' + code)
print(livy.run_statement(base_url, session_id, code))


## 4. Filter customers older than 30

Reuse the strict Age > 30 filter and select Age, Income and Family. Transformations build a plan; show/count trigger execution. Compare matching counts with Exercise 1; preview ordering is not guaranteed.

The cell displays the code being sent, followed by its remote output.

In [ ]:
heading, code = steps[1]
print('Code submitted to Livy:\n' + code)
print(livy.run_statement(base_url, session_id, code))


## 5. Count Personal Loan categories

Repeat the grouped counts ordered by raw category value. Check source documentation before interpreting codes. Counts should sum to the input count, including any null category. This is not predictive modelling.

The cell displays the code being sent, followed by its remote output.

In [ ]:
heading, code = steps[2]
print('Code submitted to Livy:\n' + code)
print(livy.run_statement(base_url, session_id, code))


## 6. Save a bounded preview locally

Collect at most ten rows at the remote driver, transfer them as JSON through
Livy, then write `outputs/02_filtered_preview.csv` in your own workspace.

Re-running this cell or the Python alternative replaces this Exercise 2 file.
Exercise 1's preview and the source CSV remain unchanged. An empty result
produces a header-only file. Do not remove the row limit; ordering is not fixed.

In [ ]:
output_file = livy.export_preview(base_url, session_id, workspace / 'outputs')
print('Saved:', output_file)


## 7. Finish — delete your Livy session

Run this cell after success or an earlier failure if a session ID was obtained.
Notebook cleanup is manual; the terminal alternative uses a finally block.

Deletion is requested for your session only. Shared containers remain running,
and an accepted request does not prove teardown has finished. If deletion
fails, the ID remains recorded for retry or reporting.

If creation timed out before returning an ID, ask the organiser to check for
an orphan session before retrying. Restarting the kernel does not delete a
remote session. Do not delete another participant's session.

### Completion checklist
- [ ] I explained where the HTTP client, driver and executors run.
- [ ] I explained session/statement IDs and asynchronous polling.
- [ ] I repeated Exercise 1's analysis and compared counts.
- [ ] I found the bounded local Exercise 2 preview.
- [ ] I requested deletion of my session.

Report missing settings, HTTP/TLS errors, remote data-path failures or cleanup
problems to the organiser. Do not guess addresses, disable TLS verification,
install dependencies or manage containers.

In [ ]:
if session_id is None:
    print('No Livy session ID is recorded in this notebook.')
else:
    livy.livy_request(base_url, 'DELETE', f'/sessions/{session_id}')
    print('Session deletion requested:', session_id)
    session_id = None
    print('Shared Livy and Spark containers remain running.')
